### Setup imports and directories
Configures data paths and loads core libraries used throughout the notebook.


In [ ]:
#imports and path 

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "../data"
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)


## Introduction & Purpose of Notebook 01

- This notebook loads raw data
- Performs EDA
- Merges datasets
- Performs feature engineering
- Produces provider-level dataset for modeling


### Load raw train/test files
Reads all train/test beneficiary, inpatient, outpatient, and label CSVs into DataFrames.


In [ ]:
#Load ALL datasets

# TRAIN FILES
train_labels = pd.read_csv("../data/Train_labels.csv")
train_inp = pd.read_csv("../data/Train_Inpatientdata.csv")
train_outp = pd.read_csv("../data/Train_Outpatientdata.csv")
train_benef = pd.read_csv("../data/Train_Beneficiarydata.csv")

# TEST FILES
test_providers = pd.read_csv("../data/Test_labels.csv")
test_inp = pd.read_csv("../data/Test_Inpatientdata.csv")
test_benef = pd.read_csv("../data/Test_Beneficiarydata.csv")
test_outp = pd.read_csv("../data/Test_Outpatientdata.csv")

# Display first few rows (check data loading)
train_labels.head(), train_inp.head(), train_outp.head(), train_benef.head(), test_providers.head(), test_inp.head(), test_benef.head(), test_outp.head()

### Data overview
Summary statistics for each key training table to understand dimensions and types before feature work.


In [ ]:
# Inspect shape, info, and descriptive stats for key training tables
datasets = {
    'Train_Inpatientdata': train_inp,
    'Train_Outpatientdata': train_outp,
    'Train_Beneficiarydata': train_benef,
    'Train_labels': train_labels,
}

for name, df in datasets.items():
    print(f"\n>>> {name} shape: {df.shape}")
    df.info()
    display(df.describe(include='all'))


### Quick EDA on raw training data
Checks dataset shapes, missing-value rates, and fraud label distribution.


In [ ]:
# Basic EDA: Check for missing values and class distribution and add plots (class distribution, claim amounts, etc.)

for name, df in [
    ("train_benef", train_benef),
    ("train_inp", train_inp),
    ("train_outp", train_outp),
    ("train_labels", train_labels),
]:
    print(f"\n{name}: shape={df.shape}")
    print(df.isnull().mean().sort_values(ascending=False).head(8))

print("\nFraud label distribution:")
print(train_labels["PotentialFraud"].value_counts(normalize=True))

### Visual EDA
Quick distributions to spot imbalance and reimbursement patterns before merging.


In [ ]:
# Simple visual EDA on key fields
sns.countplot(x=train_labels['PotentialFraud'])
plt.title('Fraud Distribution')
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(train_inp['InscClaimAmtReimbursed'].dropna(), bins=50)
plt.title('Inpatient Reimbursement Distribution')
plt.xlabel('InscClaimAmtReimbursed')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(train_outp['InscClaimAmtReimbursed'].dropna(), bins=50)
plt.title('Outpatient Reimbursement Distribution')
plt.xlabel('InscClaimAmtReimbursed')
plt.ylabel('Frequency')
plt.show()


### Inpatient feature engineering (train)
Converts dates, computes length of stay, and aggregates inpatient metrics per provider.


In [ ]:
# Convert dates and compute length of stay

train_inp["ClaimStartDt"] = pd.to_datetime(train_inp["ClaimStartDt"])
train_inp["ClaimEndDt"]   = pd.to_datetime(train_inp["ClaimEndDt"])
train_inp["LOS"] = (train_inp["ClaimEndDt"] - train_inp["ClaimStartDt"]).dt.days

inp_agg = train_inp.groupby("Provider").agg(
    inp_claims=("ClaimID", "nunique"),
    inp_total_reimbursed=("InscClaimAmtReimbursed", "sum"),
    inp_mean_reimbursed=("InscClaimAmtReimbursed", "mean"),
    inp_max_reimbursed=("InscClaimAmtReimbursed", "max"),
    inp_mean_deductible=("DeductibleAmtPaid", "mean"),
    inp_mean_los=("LOS", "mean"),
    inp_unique_bene=("BeneID", "nunique"),
).reset_index()

inp_agg.head()

### Outpatient feature engineering (train)
Aggregates outpatient claim counts, reimbursements, deductibles, and unique beneficiaries per provider.


In [ ]:
# Outpatient data aggregation

outp_agg = train_outp.groupby("Provider").agg(
    outp_claims=("ClaimID", "nunique"),
    outp_total_reimbursed=("InscClaimAmtReimbursed", "sum"),
    outp_mean_reimbursed=("InscClaimAmtReimbursed", "mean"),
    outp_mean_deductible=("DeductibleAmtPaid", "mean"),
    outp_unique_bene=("BeneID", "nunique"),
).reset_index()

outp_agg.head()

### Beneficiary feature engineering (train)
Joins beneficiary info to inpatient claims, derives ages/chronic condition indicators, and aggregates by provider.


In [ ]:
# Feature engineering on beneficiary data

train_benef["DOB"] = pd.to_datetime(train_benef["DOB"], errors="coerce")

# attach beneficiary info to inpatient claims
inp_with_bene = train_inp.merge(train_benef, on="BeneID", how="left")

# approximate age at a reference date (say 2010-01-01)
inp_with_bene["age"] = (
    (pd.Timestamp("2010-01-01") - inp_with_bene["DOB"]).dt.days / 365.25
)

# all chronic condition columns
chronic_cols = [c for c in train_benef.columns if c.startswith("ChronicCond_")]
chronic_cols

# define a helper function to compute percentage

def pct(series, predicate):
    return predicate(series).mean()

benef_agg = inp_with_bene.groupby("Provider").agg(
    avg_age=("age", "mean"),
    pct_male=("Gender", lambda x: (x == 1).mean()),
    pct_female=("Gender", lambda x: (x == 2).mean()),
    pct_renal=("RenalDiseaseIndicator", lambda x: (x == "Y").mean()),
    **{f"mean_{c}": (c, "mean") for c in chronic_cols},
).reset_index()

benef_agg.head()

### Dataset Joining Strategy
- Inpatient and outpatient claims are merged using `Provider` + `ClaimID` to align claim-level details.
- Beneficiary information is merged using `BeneID` to attach member demographics and conditions.
- Provider is used for final aggregation to build provider-level features for modeling.


### Merge training features
Combines inpatient, outpatient, and beneficiary aggregates; joins labels; creates binary fraud target.


In [ ]:
# Merge all features and prepare final training dataset

from functools import reduce

train_feature_dfs = [inp_agg, outp_agg, benef_agg]

train_features = reduce(
    lambda left, right: pd.merge(left, right, on="Provider", how="outer"),
    train_feature_dfs
)

train_full = train_features.merge(train_labels, on="Provider", how="inner")

# create binary label
train_full["FraudLabel"] = (train_full["PotentialFraud"] == "Yes").astype(int)

train_full.head(), train_full["FraudLabel"].value_counts(normalize=True)
# Save processed training data with features
train_full.to_csv(os.path.join(PROCESSED_DIR, "train_provider_features.csv"), index=False)

### Feature Engineering Explanation
- Aggregation condenses claim-level records into provider summaries for modeling efficiency.
- Provider-level features capture behavior patterns that help flag fraud risk.
- Metrics include claim counts, reimbursement totals, and unique beneficiaries treated.
- Demographic signals (age, chronic conditions) help contextualize provider patient mix.


### Inpatient feature engineering (test)
Repeats inpatient processing for the test set with LOS computation and provider-level aggregates.


In [ ]:
# Repeat feature engineering for test dataset
#Inpatient test

test_inp["ClaimStartDt"] = pd.to_datetime(test_inp["ClaimStartDt"])
test_inp["ClaimEndDt"]   = pd.to_datetime(test_inp["ClaimEndDt"])
test_inp["LOS"] = (test_inp["ClaimEndDt"] - test_inp["ClaimStartDt"]).dt.days

test_inp_agg = test_inp.groupby("Provider").agg(
    inp_claims=("ClaimID", "nunique"),
    inp_total_reimbursed=("InscClaimAmtReimbursed", "sum"),
    inp_mean_reimbursed=("InscClaimAmtReimbursed", "mean"),
    inp_max_reimbursed=("InscClaimAmtReimbursed", "max"),
    inp_mean_deductible=("DeductibleAmtPaid", "mean"),
    inp_mean_los=("LOS", "mean"),
    inp_unique_bene=("BeneID", "nunique"),
).reset_index()

### Outpatient feature engineering (test)
Aggregates outpatient claim metrics per provider for the test data.


In [ ]:
#Outpatient test

test_outp_agg = test_outp.groupby("Provider").agg(
    outp_claims=("ClaimID", "nunique"),
    outp_total_reimbursed=("InscClaimAmtReimbursed", "sum"),
    outp_mean_reimbursed=("InscClaimAmtReimbursed", "mean"),
    outp_mean_deductible=("DeductibleAmtPaid", "mean"),
    outp_unique_bene=("BeneID", "nunique"),
).reset_index()

### Beneficiary feature engineering (test)
Adds beneficiary details to test inpatient claims, derives demographics/chronic indicators, and aggregates by provider.


In [ ]:
#Beneficiary test

test_benef["DOB"] = pd.to_datetime(test_benef["DOB"], errors="coerce")

test_inp_with_bene = test_inp.merge(test_benef, on="BeneID", how="left")

test_inp_with_bene["age"] = (
    (pd.Timestamp("2010-01-01") - test_inp_with_bene["DOB"]).dt.days / 365.25
)

test_chronic_cols = [c for c in test_benef.columns if c.startswith("ChronicCond_")]

test_benef_agg = test_inp_with_bene.groupby("Provider").agg(
    avg_age=("age", "mean"),
    pct_male=("Gender", lambda x: (x == 1).mean()),
    pct_female=("Gender", lambda x: (x == 2).mean()),
    pct_renal=("RenalDiseaseIndicator", lambda x: (x == "Y").mean()),
    **{f"mean_{c}": (c, "mean") for c in test_chronic_cols},
).reset_index()

### Merge test features and save
Combines all test aggregates and writes provider-level features to processed storage.


### Summary & Export
- Final dataset contains one row per provider.
- Includes aggregated claim features across inpatient, outpatient, and beneficiary data.
- Includes the fraud target label for supervised learning.
- Will be consumed in Notebook 02 for model training.


In [ ]:
#Merge test features and save

test_feature_dfs = [test_inp_agg, test_outp_agg, test_benef_agg]

test_full = reduce(
    lambda left, right: pd.merge(left, right, on="Provider", how="outer"),
    test_feature_dfs
)

test_full.to_csv(os.path.join(PROCESSED_DIR, "test_provider_features.csv"), index=False)
test_full.head()